In [32]:
import numpy as np
import pandas as pd
import yfinance as yf

In [33]:
sector_dict = {

    "Banking": [
        "HDFCBANK.NS",
        "ICICIBANK.NS",
        "KOTAKBANK.NS",
        "SBIN.NS",
        "^NSEI"
    ],

    "IT": [
        "TCS.NS",
        "INFY.NS",
        "WIPRO.NS",
        "HCLTECH.NS",
        "^NSEI"
    ],

    "Energy": [
        "RELIANCE.NS",
        "ONGC.NS",
        "BPCL.NS",
        "^NSEI"
    ],

    "Auto": [
        "MARUTI.NS",
        "M&M.NS",
        "^NSEI"
    ],

    "FMCG": [
        "HINDUNILVR.NS",
        "ITC.NS",
        "NESTLEIND.NS",
        "^NSEI"
    ]
}

In [34]:
def download_data(tickers):
    
    data = yf.download(
        tickers,
        start="2012-01-01",
        end="2025-01-01",
        auto_adjust=False
    )
    
    data.columns = [
        f"{ticker}_{feature}"
        for feature, ticker in data.columns
    ]
    
    data = data.drop(
        columns=[
            col for col in data.columns
            if "Adj Close" in col
        ],
        errors="ignore"
    )
    
    data = data.dropna(how="all")
    
    return data

In [35]:
def select_core_columns(data):
    
    selected_cols = [
        col for col in data.columns
        if (
            "Close" in col
            or "Volume" in col
        )
    ]
    
    filtered_data = data[selected_cols]
    
    return filtered_data

In [36]:
def create_returns(data):
    
    core_data = select_core_columns(data)
    
    returns = core_data.pct_change()
    
    return returns

In [37]:
def create_momentum_features(data,windows=[3, 5, 10, 20]):
    
    core_data = select_core_columns(data)
    
    momentum_features = []
    
    for window in windows:
        
        momentum = (core_data.pct_change(window))
        
        momentum.columns = [
            f"{col}_mom_{window}"
            for col in momentum.columns
        ]
        
        momentum_features.append(momentum)
    
    momentum_df = pd.concat(momentum_features,axis=1)
    
    return momentum_df

In [38]:
def create_volatility_features(returns,windows=[5, 20]):
    
    volatility_features = []
    
    for window in windows:
        
        volatility = returns.rolling(window).std()
        
        volatility.columns = [
            f"{col}_vol_{window}"
            for col in volatility.columns
        ]
        
        volatility_features.append(volatility)
    
    volatility_df = pd.concat(
        volatility_features,
        axis=1
    )
    
    return volatility_df

In [39]:
def create_intraday_returns(data,tickers):
    
    intraday_returns = pd.DataFrame()
    
    for ticker in tickers:
        
        intraday_returns[
            f"{ticker}_intraday_return"
        ] = (
            (
                data[f"{ticker}_Close"]
                - data[f"{ticker}_Open"]
            )
            / data[f"{ticker}_Open"]
        )
    
    return intraday_returns

In [40]:
def create_hl_range(data,tickers):
    
    hl_range = pd.DataFrame()
    
    for ticker in tickers:
        
        hl_range[
            f"{ticker}_hl_range"
        ] = (
            (
                data[f"{ticker}_High"]
                - data[f"{ticker}_Low"]
            )
            / data[f"{ticker}_Close"]
        )
    
    return hl_range

In [41]:
def create_ma_distance_features(data,tickers,windows=[10, 20, 50]):
    
    ma_features = []
    
    for window in windows:
        
        ma_distance = pd.DataFrame()
        
        for ticker in tickers:
            
            ma = (
                data[f"{ticker}_Close"]
                .rolling(window)
                .mean()
            )
            
            ma_distance[
                f"{ticker}_ma_dist_{window}"
            ] = (
                (
                    data[f"{ticker}_Close"]
                    - ma
                )
                / ma
            )
        
        ma_features.append(ma_distance)
    
    ma_df = pd.concat(
        ma_features,
        axis=1
    )
    
    return ma_df

In [42]:
def create_relative_volume_features(data,tickers,window=20):
    
    relative_volume = pd.DataFrame()
    
    for ticker in tickers:
        
        if ticker == "^NSEI":
            continue
        
        volume_ma = (
            data[f"{ticker}_Volume"]
            .rolling(window)
            .mean()
        )
        
        relative_volume[
            f"{ticker}_rel_volume_{window}"
        ] = (
            data[f"{ticker}_Volume"]
            / volume_ma
        )
    
    return relative_volume

In [43]:
def create_rolling_mean_return_features(returns,windows=[5, 20]):
    
    rolling_mean_features = []
    
    for window in windows:
        
        rolling_mean = (
            returns
            .rolling(window)
            .mean()
        )
        
        rolling_mean.columns = [
            f"{col}_mean_ret_{window}"
            for col in rolling_mean.columns
        ]
        
        rolling_mean_features.append(
            rolling_mean
        )
    
    rolling_mean_df = pd.concat(
        rolling_mean_features,
        axis=1
    )
    
    return rolling_mean_df

In [44]:
def get_sector_tickers(sector_name,sector_dict):
    return sector_dict[sector_name]

In [45]:
def prepare_ticker_universe(target_ticker,sector_name,sector_dict):
    
    tickers = sector_dict[sector_name].copy()
    
    if target_ticker not in tickers:
        tickers.insert(0, target_ticker)
    
    tickers = list(set(tickers))
    
    return tickers

In [46]:
def build_feature_matrix(target_ticker,sector_name,sector_dict):
    
    
    tickers = prepare_ticker_universe(target_ticker,sector_name,sector_dict)
    
    
    data = download_data(tickers)
    
    
    returns = create_returns(data)
    
    
    momentum = create_momentum_features(data)
    
    
    volatility = create_volatility_features(returns)
    
    
    # intraday = create_intraday_returns(data,tickers)
    
    
    # hl_range = create_hl_range(data,tickers)
    
    
    ma_distance = create_ma_distance_features(data,tickers)
    
    
    # relative_volume = (create_relative_volume_features(data,tickers))
    
     
    rolling_mean = (create_rolling_mean_return_features(returns))
    
    
    all_features = pd.concat(
        [
            returns,
            momentum,
            volatility,
            ma_distance,
            rolling_mean
        ],
        axis=1)
    
    
    bad_cols = [
        col for col in all_features.columns
        if "^NSEI_Volume" in col
        or "^NSEI_rel_volume" in col
    ]

    raw_cols = [
    col for col in all_features.columns
    if col.endswith("_Close")
    or col.endswith("_Volume")
]
    

    all_features = all_features.drop(columns=raw_cols,errors="ignore")

    all_features = all_features.drop(
        columns=bad_cols,
        errors="ignore"
    )
    
    
    all_features = all_features.dropna()
    
    return all_features

In [47]:
def create_future_returns(data,target_ticker,horizon=5):
    future_returns = (
        data[f"{target_ticker}_Close"].pct_change(horizon).shift(-horizon)
    )


    future_returns = future_returns.dropna()

    return future_returns

In [48]:
def create_labels(future_returns,quantile=0.7):
    thresold = future_returns.quantile(quantile)

    labels = (
        future_returns > thresold
    ).astype(int)


    return labels

In [49]:
def prepare_dataset(target_ticker,sector_name,sector_dict,horizons=5,quantiles=0.7):
    
    features = build_feature_matrix(target_ticker,sector_name,sector_dict)

    tickers = prepare_ticker_universe(target_ticker,sector_name,sector_dict)

    data = download_data(tickers)

    future_returns = create_future_returns(data,target_ticker,horizon=horizons)

    label = create_labels(future_returns,quantiles)

    data = features.copy()

    data['Target'] = label

    data = data.dropna()

    return data 

In [69]:
dataset = prepare_dataset(
    "HDFCBANK.NS",
    "Banking",
    sector_dict
)

[*********************100%***********************]  5 of 5 completed
[*********************100%***********************]  5 of 5 completed


In [70]:
dataset['Target'].value_counts()

Target
0.0    1715
1.0     708
Name: count, dtype: int64

In [71]:
X = dataset.drop(columns=['Target'])
Y = dataset['Target']

In [72]:
from sklearn.model_selection import train_test_split

x_train,x_temp,y_train,y_temp = train_test_split(X,Y,test_size=0.3,shuffle=False)

x_val,x_test,y_val,y_test = train_test_split(x_temp,y_temp,test_size=2/3,shuffle=False)

In [73]:
print(x_train.shape)
print(x_val.shape)
print(x_test.shape)

(1696, 87)
(242, 87)
(485, 87)


In [74]:
x_train_final = pd.concat([x_train, x_val])
y_train_final = pd.concat([y_train, y_val])

In [75]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score,classification_report,confusion_matrix)

In [76]:
pipeline1 = Pipeline(
    steps = [
    ('scaler',StandardScaler()),
    ('model',LogisticRegression(max_iter=1000, class_weight='balanced'))
    ]
)

In [83]:
pipeline2 = Pipeline(
    steps = [
        ('xgb_model',XGBClassifier(max_depth = 5,
        learning_rate = 0.008,
        n_estimators = 10,
        scale_pos_weight = y_train_final.value_counts()[0]/y_train_final.value_counts()[1]))
    ]
)

In [84]:
pipeline1.fit(x_train_final,y_train_final)

y_pred_lr = pipeline1.predict(x_test)

print(classification_report(y_test,y_pred_lr))

              precision    recall  f1-score   support

         0.0       0.74      0.64      0.69       352
         1.0       0.30      0.41      0.35       133

    accuracy                           0.58       485
   macro avg       0.52      0.53      0.52       485
weighted avg       0.62      0.58      0.60       485



In [85]:
pipeline2.fit(x_train_final,y_train_final)

y_pred_xgb = pipeline2.predict(x_test)

print(classification_report(y_test,y_pred_xgb))

              precision    recall  f1-score   support

         0.0       0.70      0.59      0.64       352
         1.0       0.24      0.34      0.28       133

    accuracy                           0.52       485
   macro avg       0.47      0.46      0.46       485
weighted avg       0.58      0.52      0.54       485



In [86]:
results = pd.DataFrame({
    "Stock": ["HDFCBANK", "TCS", "MARUTI", "RELIANCE"],
    "LR_F1": [0.35, 0.38, 0.24, 0.23],
    "XGB_F1": [0.28, 0.37, 0.32, 0.26]
})

results

,Stock,LR_F1,XGB_F1
0,HDFCBANK,0.35,0.28
1,TCS,0.38,0.37
2,MARUTI,0.24,0.32
3,RELIANCE,0.23,0.26


In [87]:
# import joblib

# joblib.dump(pipeline1,'logistic_regression.pkl')
# joblib.dump(pipeline2,'xgboost.pkl')